In [1]:
# imports

import numpy as np
import re
import igraph as ig
import pandas as pd
import scanpy as sc

import anndata as ad
from sklearn.preprocessing import StandardScaler

import utils as ut
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
np.random.seed(42)

## Helper functions

In [2]:
## Helper functions
def double_z_norm(data):
    scaler = StandardScaler()
    step1 = scaler.fit_transform(data)

    # Then z-score across samples (rows)
    scaler2 = StandardScaler()
    normalised_data = scaler2.fit_transform(step1.T).T
    return normalised_data

In [3]:


def subcluster_multiple_builtin(
    adata, 
    cluster_key='leiden', 
    clusters_to_split=['0', '3', '5'], 
    resolutions=None,
    key_added='leiden_subclustered'
):
    """
    Subcluster multiple clusters individually using built-in restrict_to,
    and merge all results into a single annotation column.
    """
    # Handle resolution parameter
    if resolutions is None:
        resolutions = {c: 0.5 for c in clusters_to_split}
    elif isinstance(resolutions, (int, float)):
        resolutions = {c: resolutions for c in clusters_to_split}
    
    # Initialize with original clusters
    adata.obs[key_added] = adata.obs[cluster_key].astype(str).copy()
    
    # Subcluster each target cluster
    for cluster_id in clusters_to_split:
        cluster_id_str = str(cluster_id)
        resolution = resolutions.get(cluster_id_str, 0.5)
        
        print(f"Subclustering cluster {cluster_id_str} with resolution {resolution}")
        
        # Use built-in restrict_to for this cluster
        temp_key = f'_temp_sub_{cluster_id_str}'
        sc.tl.leiden(
            adata,
            restrict_to=(cluster_key, [cluster_id_str]),
            resolution=resolution,
            key_added=temp_key
        )
        
        # Extract subclusters for this cluster only
        mask = adata.obs[cluster_key].astype(str) == cluster_id_str
        
        # Get the subcluster part and relabel as 'X.0', 'X.1', 'X.2'
        subclusters = adata.obs.loc[mask, temp_key].astype(str)
        new_labels = subclusters.str.replace(',', '.')
        
        # Update the final annotation
        adata.obs.loc[mask, key_added] = new_labels
        
        # Clean up temp column
        adata.obs.drop(columns=[temp_key], inplace=True)
        
        n_subclusters = len(adata.obs.loc[mask, key_added].unique())
        print(f"  → Split into {n_subclusters} subclusters: {sorted(adata.obs.loc[mask, key_added].unique())}")
    
    # Convert to categorical
    adata.obs[key_added] = adata.obs[key_added].astype('category')
    
    print(f"\nFinal clusters: {sorted(adata.obs[key_added].unique())}")
    
    return adata


def remap_cluster_labels(adata, cluster_key, label_mapping, new_key=None):
    """
    Remap cluster labels to custom annotations.
    
    Parameters:
    -----------
    adata : AnnData
        Annotated data object
    cluster_key : str
        Column name containing clusters to remap
    label_mapping : dict
        Mapping from old labels to new labels
        Example: {'0': 'T cells', '1': 'B cells', '0.0': 'CD4 T cells'}
    new_key : str, optional
        Name for new column. If None, updates cluster_key in place
    
    Returns:
    --------
    adata : AnnData
        Updated AnnData object
    """
    if new_key is None:
        # Rename in place
        adata.obs[cluster_key] = adata.obs[cluster_key].astype(str).map(label_mapping)
        
        # Handle any unmapped values
        if adata.obs[cluster_key].isna().any():
            print(f"Warning: {adata.obs[cluster_key].isna().sum()} cells could not be mapped")
            print(f"Unmapped original values: {adata.obs[adata.obs[cluster_key].isna()][cluster_key].unique()}")
        
        adata.obs[cluster_key] = adata.obs[cluster_key].astype('category')
    else:
        # Create new column
        adata.obs[new_key] = adata.obs[cluster_key].astype(str).map(label_mapping)
        
        # Handle unmapped values
        if adata.obs[new_key].isna().any():
            print(f"Warning: {adata.obs[new_key].isna().sum()} cells could not be mapped")
            # Keep original label for unmapped
            mask = adata.obs[new_key].isna()
            adata.obs.loc[mask, new_key] = adata.obs.loc[mask, cluster_key].astype(str)
        
        adata.obs[new_key] = adata.obs[new_key].astype('category')
    
    print(f"Remapped clusters: {sorted(adata.obs[cluster_key if new_key is None else new_key].unique())}")
    
    return adata


def plot_cluster_analysis(adata, var_names, groupby, save_prefix='cluster_analysis'):
    """
    Generate dotplot and heatmap for cluster analysis.
    
    Parameters:
    -----------
    adata : AnnData
        Annotated data object
    var_names : list
        List of marker genes/features to plot
    groupby : str
        Column name containing cluster annotations
    save_prefix : str
        Prefix for saved figure files
    """
    # Dotplot
    print(f"Generating dotplot for {groupby}...")
    sc.pl.dotplot(
        adata=adata,
        var_names=var_names,
        groupby=groupby,
        dendrogram=True,
        save=f'_{save_prefix}_dotplot.png'
    )
    
    # Heatmap
    print(f"Generating heatmap for {groupby}...")
    sc.pl.heatmap(
        adata=adata,
        var_names=var_names,
        groupby=groupby,
        dendrogram_key=groupby,
        swap_axes=True,
        cmap='RdYlBu_r',
        save=f'_{save_prefix}_heatmap.png'
    )
    
    print(f"Plots saved with prefix: {save_prefix}")

## Setup

In [4]:

INPUT_DIR = "./data/raw/mrd"
OUTPUT_DIR = "./data/output/"

np.random.seed(42)

# Step 1: Load the CSV file
# df = pd.read_csv("8_SN141_slides2_ROI-03.csv")

import os

feature_files = []
region_list = []
for path, subdirs, files in os.walk(INPUT_DIR):
    for name in files:
        fname, ext = os.path.splitext(name)
        if ext == ".csv":
            rel_path = os.path.relpath(path, INPUT_DIR)
            feature_files.append(os.path.join(rel_path, name))
            region_list.append(os.path.join(rel_path, fname))
print(feature_files)
print(region_list)

df = ut.combine_feature_files(
    INPUT_DIR,
    feature_files,
    region_list
    )


['./7_SN141_slides1_ROI-07.csv', './3_SN177_ROI-09.csv', './10_SN141_slides2_ROI-16.csv', './6_SN177_ROI-16.csv', './1_SN177_ROI-06.csv', './8_SN141_slides2_ROI-03.csv', './2_SN177_ROI-07.csv', './5_SN177_ROI-14.csv', './9_SN141_slides2_ROI-06.csv', './4_SN177_ROI-13.csv']
['./7_SN141_slides1_ROI-07', './3_SN177_ROI-09', './10_SN141_slides2_ROI-16', './6_SN177_ROI-16', './1_SN177_ROI-06', './8_SN141_slides2_ROI-03', './2_SN177_ROI-07', './5_SN177_ROI-14', './9_SN141_slides2_ROI-06', './4_SN177_ROI-13']
0
1
2
3
4
5
6
7
8
9
Invalid columns: ['DAPI C0 Biomarker Exp', 'DAPI C1 Biomarker Exp', 'DAPI C2 Biomarker Exp', 'DAPI C3 Biomarker Exp', 'DAPI C4 Biomarker Exp', 'DAPI C5 Biomarker Exp', 'DAPI C6 Biomarker Exp', 'DAPI C7 Biomarker Exp', 'DAPI C8 Biomarker Exp', 'DAPI C9 Biomarker Exp', 'DAPI C10 Biomarker Exp', 'DAPI C11 Biomarker Exp', 'DAPI C12 Biomarker Exp', 'DAPI C13 Biomarker Exp', 'DAPI C14 Biomarker Exp', 'DAPI C15 Biomarker Exp', 'DAPI C16 Biomarker Exp', 'DAPI C17 Biomarker Ex

In [5]:
meta_column_names = ["Cell Id","Nuc X","Nuc Y Inv", "region_num", "unique_region"]

selected_features = [
"Arginase 1 REAL1137",
"CD11b REA1321",
"CD11c REA1310",
"CD14 REA1314",
"CD15 VIMC6",
"CD16 REA1324",
"CD163 REA1309",
"CD20 REAL1069",
"CD235a REA175",
"CD3 REAL1097",
"CD31 REA1312",
"CD34 REAL1217",
"CD38 REAL719",
"CD4 REA1307",
"CD45 5B1",
"CD68 REAL1346",
"CD73 REAL1172",
"CD79a REA1168",
"CD8a REA1024",
"FoxP3 REA1253",
"HLA DR REAL550",
"Ki 67 REAL1047",
"Mast Cell Tryptase REAL798",
"PAX 5 REA140",
"Podoplanin REA446",
]

In [6]:
# 0.b clean column names
df.columns = df.columns.str.replace('Biomarker Exp', '', regex=False).str.strip()

# Step 2: Separate features and metadata
feature_cols = df.columns.difference(meta_column_names)
unselected_features = feature_cols.difference(selected_features)
unselected_feat_exp = df[unselected_features].copy()
selected_feat_exp = df[selected_features].copy()
metadata = df[df.columns.difference(selected_features)].copy()

display(selected_feat_exp)
display(metadata)

,Arginase 1 REAL1137,CD11b REA1321,CD11c REA1310,CD14 REA1314,CD15 VIMC6,CD16 REA1324,CD163 REA1309,CD20 REAL1069,CD235a REA175,CD3 REAL1097,...,CD68 REAL1346,CD73 REAL1172,CD79a REA1168,CD8a REA1024,FoxP3 REA1253,HLA DR REAL550,Ki 67 REAL1047,Mast Cell Tryptase REAL798,PAX 5 REA140,Podoplanin REA446
0,92.446075,142.259811,263.029419,260.794128,473.009796,82.500000,129.750000,32.166668,33.583332,199.784317,...,101.857841,232.024506,50.289215,50.299019,28.663794,606.529419,2906.741455,915.549744,30.655172,18.553921
1,63.928825,81.206406,150.805450,102.587189,215.425858,116.973900,128.488724,21.953737,291.067627,21.415184,...,53.959667,192.825623,22.043890,41.408066,27.837742,101.887306,11579.800781,395.544678,15.996472,13.302491
2,58.732277,100.416931,155.667725,154.360840,240.758728,184.276184,247.616928,20.578836,34.892063,27.311111,...,51.900528,353.150269,22.980953,43.705822,23.393898,358.539673,494.533966,334.992188,20.898752,16.136507
3,100.900002,565.174622,352.888885,518.606323,901.536499,260.147614,384.165070,52.080952,35.563492,98.179367,...,75.065079,315.469849,46.155556,86.341270,25.766270,789.976196,3433.557373,625.216736,24.136572,44.326984
4,152.121292,192.881485,293.547211,368.655548,471.563904,323.624084,1158.716675,35.507408,38.294445,92.830559,...,98.732407,464.384247,70.951851,75.044441,29.427341,264.619446,9817.036133,710.422852,19.053537,27.427778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59097,376.449432,394.325836,134.213486,160.157303,10879.831055,64.168541,140.606735,39.853931,64.303368,30.943821,...,51.325844,1152.561768,10.359550,902.191040,38.985176,2506.078613,8195.530273,2444.791748,18.120296,31.033709
59098,52.661228,45.853199,109.343788,117.120453,486.233368,60.086575,498.111664,33.441658,333.089081,45.767879,...,266.397736,2092.158203,12.819323,2602.563477,38.961128,5011.687500,2323.346924,2841.781494,19.980564,84.668755
59099,25.795990,25.566628,57.830780,44.235260,179.464630,25.951651,133.029480,16.670401,43.293041,16.359079,...,84.324295,450.665680,10.929245,148.401535,34.485733,122.227005,690.050171,862.170654,10.052939,16.146816
59100,66.221237,206.017700,77.504425,230.247787,1534.725708,69.088493,203.123901,22.212389,43.769913,17.743362,...,47.053097,399.743378,9.752213,75.176994,39.306450,4681.858398,1326.938477,1031.706787,19.085686,22.982302


,AKT Pan REA676,Actin REAL650,Alexa647 anti rabbit1,BATF REA486,Bcl 2 REA872,CD104 REA236,CD117 REA787,CD123 REA918,CD134 ACT35,CD147 REA282,...,RRM2 REAL1010,Synaptophysin REA1121,TIM 3 REAL818,WT1 REA925,ZAP70 REA814,beta Catenin REA480,c myc REAL810,p53 REA1132,region_num,unique_region
0,30.198277,411.076538,298.543365,24.584482,581.000000,23.480392,20.686274,15.156863,25.730392,1141.382324,...,9.328431,61.014706,22.897058,24.214285,40.127453,53.101723,827.463013,39.113792,0,./7_SN141_slides1_ROI-07
1,36.241623,383.353912,236.329071,22.696650,875.947083,22.534994,25.661922,20.035587,23.281139,567.376038,...,11.513641,43.565838,19.298933,25.024113,16.855278,73.231041,592.463135,80.684303,0,./7_SN141_slides1_ROI-07
2,30.219141,328.591248,63.816326,21.246880,783.839111,20.341799,23.141798,12.034921,23.593651,444.347076,...,9.282539,37.395767,23.786243,25.612844,42.633862,11.475728,507.073822,107.355064,0,./7_SN141_slides1_ROI-07
3,26.887259,476.370728,215.096451,24.837763,513.689270,21.661905,28.871429,20.187302,23.601587,713.099976,...,12.411111,69.365082,36.380951,28.993608,62.971428,120.105408,754.982544,57.815765,0,./7_SN141_slides1_ROI-07
4,33.850861,536.518372,245.143463,26.289675,601.591797,20.555555,25.977777,22.016666,23.535185,711.905579,...,10.722222,66.894447,38.243519,28.908278,23.784260,34.335564,861.088928,73.157745,0,./7_SN141_slides1_ROI-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59097,34.911060,1283.616333,77.422134,43.411629,641.135132,53.876404,32.764046,15.146068,45.213482,486.808990,...,102.966293,45.213482,68.224716,23.269669,49.696629,16.689852,897.308167,55.365452,9,./4_SN177_ROI-13
59098,34.062195,1292.400879,61.743702,49.239067,493.184631,114.693855,39.023838,17.939774,41.370136,347.888336,...,143.958588,64.271019,126.634880,21.675795,80.869514,13.215743,1174.650024,98.710396,9,./4_SN177_ROI-13
59099,31.336885,323.088379,45.877960,51.517017,350.674805,33.759434,31.873232,9.711675,32.868515,134.290100,...,18.679245,17.283018,29.724056,15.748968,20.089033,13.216569,388.950714,67.723274,9,./4_SN177_ROI-13
59100,31.754032,619.427124,68.266968,41.565525,710.777222,27.654867,35.017700,15.752213,34.433628,255.495575,...,29.176991,20.221239,25.911505,21.263348,25.495575,13.935484,772.108582,45.939518,9,./4_SN177_ROI-13


In [7]:
# Step 3: Scale and normalise
scaled_features = double_z_norm(selected_feat_exp)
scaled_unselected_features = double_z_norm(unselected_feat_exp)

In [8]:
# Step 4: Create an AnnData object
adata = ad.AnnData(X=scaled_features)
adata.var_names = selected_features
adata.obs = metadata
display(adata)

AnnData object with n_obs × n_vars = 59102 × 25
    obs: 'AKT Pan REA676', 'Actin REAL650', 'Alexa647 anti rabbit1', 'BATF REA486', 'Bcl 2 REA872', 'CD104 REA236', 'CD117 REA787', 'CD123 REA918', 'CD134 ACT35', 'CD147 REA282', 'CD181 REA958', 'CD182 REA208', 'CD183 REAL756', 'CD195 REA245', 'CD196 REA190', 'CD1c REAL1005', 'CD2 REA1130', 'CD202b REA198', 'CD204 REA460', 'CD209 REAL690', 'CD223 REA351', 'CD226 REA1040', 'CD23 REAL1096', 'CD244 REAL1225', 'CD247 REAL1135', 'CD271 REAL709', 'CD274 REA1308', 'CD279 PD1 3 1 3', 'CD295 REA361', 'CD305 REA447', 'CD317 REA202', 'CD43 REA833', 'CD44 REA690', 'CD45RA REAL164', 'CD45RO REA611', 'CD52 REA164', 'CD56 REAL1142', 'CD57 REA769', 'CD88 REA1213', 'CD90 REAL677', 'CD99 REA1174', 'Cell Id', 'EZH2 REA907', 'EmptyPE1', 'FAK pS910 REA407', 'FcepsilonRIalpha REA758', 'Galectin 9 REA435', 'IFN gamma REAL788', 'IRF 7 REA521', 'JNK1 REAL1128', 'JNK2 REA1153', 'NPM1mut primaryRab400 polyclonal', 'Nuc X', 'Nuc Y Inv', 'Nucleophosmin REAL823', 'PCN

## Complete Workflow

In [ ]:

# 1. Initial clustering
print("Step 1: Initial clustering")
sc.pp.neighbors(adata, n_neighbors=9, n_pcs=30)
sc.tl.leiden(adata, resolution=1.0, key_added='leiden')

# Inspect cluster assignments
display(adata.obs['leiden'].value_counts())

display(adata)

# Visualize initial clustering
# sc.pl.umap(adata, color='leiden', legend_loc='on data')


Step 1: Initial clustering


/tmp/ipykernel_1267/1027285163.py:4: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=1.0, key_added='leiden')


leiden
0     4560
1     4481
2     3954
3     3666
4     3370
5     3326
6     3319
7     3074
8     3048
9     2701
10    2587
11    2491
12    2335
13    2018
14    1987
15    1811
16    1457
17    1151
18    1100
19    1053
20     937
21     907
22     795
23     657
24     546
25     505
26     393
27     370
28     279
29     214
30      10
Name: count, dtype: int64

AnnData object with n_obs × n_vars = 59102 × 25
    obs: 'AKT Pan REA676', 'Actin REAL650', 'Alexa647 anti rabbit1', 'BATF REA486', 'Bcl 2 REA872', 'CD104 REA236', 'CD117 REA787', 'CD123 REA918', 'CD134 ACT35', 'CD147 REA282', 'CD181 REA958', 'CD182 REA208', 'CD183 REAL756', 'CD195 REA245', 'CD196 REA190', 'CD1c REAL1005', 'CD2 REA1130', 'CD202b REA198', 'CD204 REA460', 'CD209 REAL690', 'CD223 REA351', 'CD226 REA1040', 'CD23 REAL1096', 'CD244 REAL1225', 'CD247 REAL1135', 'CD271 REAL709', 'CD274 REA1308', 'CD279 PD1 3 1 3', 'CD295 REA361', 'CD305 REA447', 'CD317 REA202', 'CD43 REA833', 'CD44 REA690', 'CD45RA REAL164', 'CD45RO REA611', 'CD52 REA164', 'CD56 REAL1142', 'CD57 REA769', 'CD88 REA1213', 'CD90 REAL677', 'CD99 REA1174', 'Cell Id', 'EZH2 REA907', 'EmptyPE1', 'FAK pS910 REA407', 'FcepsilonRIalpha REA758', 'Galectin 9 REA435', 'IFN gamma REAL788', 'IRF 7 REA521', 'JNK1 REAL1128', 'JNK2 REA1153', 'NPM1mut primaryRab400 polyclonal', 'Nuc X', 'Nuc Y Inv', 'Nucleophosmin REAL823', 'PCN

KeyError: "Could not find 'umap' or 'X_umap' in .obsm"

In [11]:
# 2. Subcluster selected clusters
print("\nStep 2: Subclustering")
adata = subcluster_multiple_builtin(
    adata,
    cluster_key='leiden',
    clusters_to_split=['0', '3', '5'],
    resolutions={
        '0': 0.8,
        '3': 0.6,
        '5': 0.5
    },
    key_added='leiden_refined'
)

# Visualize subclustering results
# sc.pl.umap(adata, color=['leiden', 'leiden_refined'], legend_loc='on data')

# 3. Remap cluster labels to biological annotations
print("\nStep 3: Label remapping")
label_mapping = {
    # Original clusters that weren't subclustered
    '1': 'B cells',
    '2': 'Monocytes',
    '4': 'NK cells',
    '6': 'Dendritic cells',
    '7': 'Platelets',
    
    # Subclusters of cluster 0
    '0.0': 'CD4 T cells',
    '0.1': 'CD8 T cells',
    '0.2': 'Regulatory T cells',
    
    # Subclusters of cluster 3
    '3.0': 'Classical Monocytes',
    '3.1': 'Non-classical Monocytes',
    
    # Subclusters of cluster 5
    '5.0': 'Naive B cells',
    '5.1': 'Memory B cells',
}

adata = remap_cluster_labels(
    adata,
    cluster_key='leiden_refined',
    label_mapping=label_mapping,
    new_key='cell_type'
)

# Visualize with new labels
# sc.pl.umap(adata, color='cell_type', legend_loc='on data')

# 4. Define marker genes for plotting
selected_features = [
    'CD3', 'CD4', 'CD8', 'CD19', 'CD20',  # Lymphocyte markers
    'CD14', 'CD16', 'CD56',                # Monocyte/NK markers
    'CD45', 'HLA-DR'                       # General markers
]

# 5. Generate plots with original subclustered labels
print("\nStep 4: Generating plots for leiden_refined")
plot_cluster_analysis(
    adata,
    var_names=selected_features,
    groupby='leiden_refined',
    save_prefix='leiden_refined'
)

# 6. Generate plots with biological annotations
print("\nStep 5: Generating plots for cell_type")
plot_cluster_analysis(
    adata,
    var_names=selected_features,
    groupby='cell_type',
    save_prefix='cell_type'
)

# 7. Optional: Reorder categories for better visualization
print("\nStep 6: Reordering categories")
desired_order = [
    'CD4 T cells', 'CD8 T cells', 'Regulatory T cells',
    'Naive B cells', 'Memory B cells', 'B cells',
    'Classical Monocytes', 'Non-classical Monocytes', 'Monocytes',
    'NK cells', 'Dendritic cells', 'Platelets'
]

# Filter to only include categories that exist
existing_order = [c for c in desired_order if c in adata.obs['cell_type'].cat.categories]
adata.obs['cell_type'] = adata.obs['cell_type'].cat.reorder_categories(existing_order)

# Replot with ordered categories
print("\nStep 7: Generating plots with ordered categories")
plot_cluster_analysis(
    adata,
    var_names=selected_features,
    groupby='cell_type',
    save_prefix='cell_type_ordered'
)

# 8. Summary statistics
print("\nCluster Summary:")
print(adata.obs['cell_type'].value_counts())

print("\nWorkflow complete!")


Step 2: Subclustering
Subclustering cluster 0 with resolution 0.8
  → Split into 13 subclusters: ['0.0', '0.1', '0.10', '0.11', '0.12', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9']
Subclustering cluster 3 with resolution 0.6
  → Split into 11 subclusters: ['3.0', '3.1', '3.10', '3.2', '3.3', '3.4', '3.5', '3.6', '3.7', '3.8', '3.9']
Subclustering cluster 5 with resolution 0.5
  → Split into 8 subclusters: ['5.0', '5.1', '5.2', '5.3', '5.4', '5.5', '5.6', '5.7']

Final clusters: ['0.0', '0.1', '0.10', '0.11', '0.12', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3.0', '3.1', '3.10', '3.2', '3.3', '3.4', '3.5', '3.6', '3.7', '3.8', '3.9', '30', '4', '5.0', '5.1', '5.2', '5.3', '5.4', '5.5', '5.6', '5.7', '6', '7', '8', '9']

Step 3: Label remapping
Remapped clusters: ['0.10', '0.11', '0.12', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9

KeyError: "Could not find keys [np.str_('CD14'), np.str_('CD16'), np.str_('CD19'), np.str_('CD20'), np.str_('CD3'), np.str_('CD4'), np.str_('CD45'), np.str_('CD56'), np.str_('CD8'), np.str_('HLA-DR')] in columns of `adata.obs` or in adata.var_names."